In [1]:
# ============================================================
# Notebook    : 08a_case_c_lightgbm_static.ipynb
# Description : Case C — Model C1 LightGBM-Static
#               Static variables only (9 features), NO BonusMalus.
#               Row-level stratified split (70/15/15, seed=42) —
#               same discipline as Case B (06a/06b). freMTPL2 is
#               confirmed genuinely cross-sectional (CHECK 2 in
#               notebook 07 — every IDpol appears exactly once),
#               so no sequence-leakage concern exists here.
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install lightgbm scikit-learn pandas numpy


# ============================================================
# 1. Common imports
# ============================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

SEED = 42
np.random.seed(SEED)


# ============================================================
# 2. Load preprocessed data (from notebook 07)
# ============================================================
df = pd.read_csv("data/fremtpl2_preprocessed.csv")
print(f"[CHECK 1] df shape: {df.shape}")
print(f"[CHECK 1] Positive rate: {df['Label'].mean()*100:.2f}%")


# ============================================================
# 3. Feature scope — C1 (static only, no BonusMalus)
# ============================================================
NUMERIC_COLS = ["Exposure", "VehPower", "VehAge", "DrivAge", "Density"]
CAT_COLS     = ["VehBrand", "VehGas", "Area", "Region"]
FEATURE_COLS = NUMERIC_COLS + CAT_COLS
LABEL_COL    = "Label"

print(f"\n[CHECK 2] Feature scope (C1 — static only, {len(FEATURE_COLS)} features):")
print(f"  Numeric     : {NUMERIC_COLS}")
print(f"  Categorical : {CAT_COLS}")


# ============================================================
# 4. Train / val / test split — row-level, stratified,
#    70/15/15, seed=42 (identical discipline to Case B)
# ============================================================
X = df[FEATURE_COLS].copy()
y = df[LABEL_COL].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"\n[CHECK 3] Split sizes — train: {len(X_train):,}, "
      f"val: {len(X_val):,}, test: {len(X_test):,}")
print(f"[CHECK 3] Positive rates — "
      f"train: {y_train.mean()*100:.2f}%, "
      f"val: {y_val.mean()*100:.2f}%, "
      f"test: {y_test.mean()*100:.2f}%")


# ============================================================
# 5. Categorical dtype handling
# ============================================================
for col in CAT_COLS:
    X_train[col] = X_train[col].astype("category")
    X_val[col]   = X_val[col].astype("category")
    X_test[col]  = X_test[col].astype("category")

    all_cats = pd.api.types.union_categoricals(
        [X_train[col], X_val[col], X_test[col]]
    ).categories
    X_train[col] = X_train[col].cat.set_categories(all_cats)
    X_val[col]   = X_val[col].cat.set_categories(all_cats)
    X_test[col]  = X_test[col].cat.set_categories(all_cats)

print(f"\n[CHECK 4] X_train shape: {X_train.shape}")


# ============================================================
# 6. Train LightGBM
# ============================================================
POS_RATE = y_train.mean()
SCALE_POS_WEIGHT = (1 - POS_RATE) / POS_RATE
print(f"\n[CHECK 5] scale_pos_weight: {SCALE_POS_WEIGHT:.2f}")

train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=CAT_COLS)
val_set   = lgb.Dataset(X_val,   label=y_val,   categorical_feature=CAT_COLS, reference=train_set)

params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "seed": SEED,
    "verbose": -1,
}

print(f"[CHECK 5] Training LightGBM-Static (C1)...")
model = lgb.train(
    params,
    train_set,
    num_boost_round=500,
    valid_sets=[train_set, val_set],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=50),
    ],
)
print(f"\n[CHECK 5] Best iteration: {model.best_iteration}")


# ============================================================
# 7. Evaluate
# ============================================================
y_pred_prob = model.predict(X_test, num_iteration=model.best_iteration)
y_pred_cls  = (y_pred_prob >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_prob)
f1  = f1_score(y_test, y_pred_cls)

precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_prob)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx  = np.argmax(f1_scores[:-1])

print("\n===== Test set evaluation =====")
print(f"Total test rows   : {len(y_test):,}")
print(f"AUC-ROC          : {auc:.4f}")
print(f"F1 @ thr=0.5      : {f1:.4f}")
print(f"Best threshold    : {thresholds[best_idx]:.3f}")
print(f"Best F1           : {f1_scores[best_idx]:.4f}")
print(f"  Precision       : {precisions[best_idx]:.4f}")
print(f"  Recall          : {recalls[best_idx]:.4f}")
print(f"Positive rate in test: {y_test.mean()*100:.2f}%")


# ============================================================
# 8. Feature importance
# ============================================================
importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "gain": model.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)

print("\n===== Feature importance (gain) =====")
print(importance.to_string(index=False))


# ============================================================
# 9. Save
# ============================================================
model.save_model("data/sequences/case_c_lightgbm_static_model.txt")
np.savez(
    "data/sequences/case_c_lightgbm_static_test_predictions.npz",
    probs=y_pred_prob, labels=y_test.values,
)
print("\nSaved model and predictions.")


# ============================================================
# 10. Summary
# ============================================================
print("\n===== LightGBM-Static (Model C1) Summary =====")
print(f"Feature scope        : {len(FEATURE_COLS)} features (static, no BonusMalus)")
print(f"Train / Val / Test   : {len(X_train):,} / {len(X_val):,} / {len(X_test):,}")
print(f"Best iteration       : {model.best_iteration}")
print(f"Test AUC-ROC         : {auc:.4f}")
print(f"Test F1 (best thr)   : {f1_scores[best_idx]:.4f}")
print("===============================================")

[CHECK 1] df shape: (678013, 13)
[CHECK 1] Positive rate: 5.02%

[CHECK 2] Feature scope (C1 — static only, 9 features):
  Numeric     : ['Exposure', 'VehPower', 'VehAge', 'DrivAge', 'Density']
  Categorical : ['VehBrand', 'VehGas', 'Area', 'Region']

[CHECK 3] Split sizes — train: 474,609, val: 101,702, test: 101,702
[CHECK 3] Positive rates — train: 5.02%, val: 5.02%, test: 5.02%

[CHECK 4] X_train shape: (474609, 9)

[CHECK 5] scale_pos_weight: 18.91
[CHECK 5] Training LightGBM-Static (C1)...
Training until validation scores don't improve for 30 rounds
[50]	train's auc: 0.687091	val's auc: 0.679802
[100]	train's auc: 0.697417	val's auc: 0.682676
[150]	train's auc: 0.705216	val's auc: 0.683528
[200]	train's auc: 0.71208	val's auc: 0.684066
Early stopping, best iteration is:
[217]	train's auc: 0.714588	val's auc: 0.68428

[CHECK 5] Best iteration: 217

===== Test set evaluation =====
Total test rows   : 101,702
AUC-ROC          : 0.6788
F1 @ thr=0.5      : 0.1418
Best threshold    : 0